In [47]:
import pandas as pd
import numpy as np
import gffutils
from collections import defaultdict

main_results = pd.read_csv("/home/marc/projects/rna_seq_workflow/results/DEG_analysis/FOXG_ASM14995v2/FOXG_ASM14995v2_main_results.csv")
eggnog_results = pd.read_csv("/home/marc/projects/rna_seq_workflow/results/functional_annotations/MM_fezatkf9.emapper.annotations.tsv", 
                            sep="\t",
                            header=4,
                            index_col=0,
                            skipfooter=3)

main_results = main_results.set_index("protein_ids")

eggnog_results = eggnog_results[["Description","KEGG_ko"]]
eggnog_results = eggnog_results.rename(columns={"Description": "eggNOG_description",})

main_results = main_results.join(eggnog_results)

interpro_gff = gffutils.FeatureDB("/home/marc/projects/rna_seq_workflow/resources/FOXG_ASM14995v2_interpro_results_db")

# interpro_results = {}
source_desc = {}
interpro_results = defaultdict(dict)
# Iterate through the features in the database
for feature in interpro_gff.all_features():
  # Check if the 'Target'(=protein id) and 'signature_desc'(=protein description) attributes exist
  if 'Target' in feature.attributes and 'signature_desc' in feature.attributes:
    #Source = Database
    source = feature.source
    descriptions = feature.attributes['signature_desc']
    targets = feature.attributes['Target']
    #Split the protein id from the specified location coordinates
    targets = targets[0].split(" ")[0]
    #Check if source already exists in the nested dict as subkey
    if source in interpro_results[targets].keys() and not "MobiDBLite":
      #If it exists add the new descritption as value, turn it into a set to remove duplicates and back into a list 
      interpro_results[targets][source] = list(set(interpro_results[targets][source]+descriptions))
    else:
      #If it doesn't exist add it with its corresponding descritption as value
      interpro_results[targets][source] = descriptions

interpro_results = dict(interpro_results)
interpro_results_df = pd.DataFrame.from_dict(interpro_results, orient="index")

main_results = main_results.join(interpro_results_df)
main_results = main_results.sort_values(by=["log2FC_shrinked","padj"], ascending=False)

main_results.to_csv("/home/marc/projects/rna_seq_workflow/results/functional_annotations/main_results_annotated.csv")


/tmp/ipykernel_1575/1643224761.py:7: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support skipfooter; you can avoid this warning by specifying engine='python'.
  eggnog_results = pd.read_csv("/home/marc/projects/rna_seq_workflow/results/functional_annotations/MM_fezatkf9.emapper.annotations.tsv",
